In [3]:
import sympy as sp
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy import special
import math
import pandas as pd

def moshinsky_function(x, k, t, m, hbar):
    M = (1/2) * special.erfc(
        (x - (hbar * k * t / m)) / (np.sqrt(2 * hbar * 1j * t / m))
    ) * np.exp(
        1j * (k * x - hbar * (k**2) * t / (2 * m))
    )
    return M

hbar = 6.582119e-1
V0 = 0.1
a = 500
c = 3e8
x = 500
E0 = 0.001
m = 0.067 * 1e10 * 0.510998e6 / c**2
m_e = 0.067 * 5.6791e-8

K0 = np.sqrt((2 * m * E0) / (hbar**2))
Kv0 = np.sqrt((2 * m * V0) / (hbar**2))
Alpha = a * Kv0
Gamma = math.tanh(Alpha) * (Kv0 / 2)
Tk = (K0**2) / (math.cosh(Alpha)**2 * (K0**2 + Gamma**2))

omega = 0.23
V_positive = V0 - hbar * omega / 2
V_negative = V0 + hbar * omega / 2
Kv0_positive = np.sqrt((2 * m * V_positive) / (hbar**2))
Kv0_negative = np.sqrt((2 * m * V_negative) / (hbar**2))
Alpha_positive = a * Kv0_positive
Alpha_negative = a * Kv0_negative
Gamma_positive = math.tanh(Alpha_positive) * Kv0_positive / 2
Gamma_negative = math.tanh(Alpha_negative) * Kv0_negative / 2

Tk_up = (K0**2) / (math.cosh(Alpha_positive)**2 * (K0**2 + Gamma_positive**2))
Tk_down = (K0**2) / (math.cosh(Alpha_negative)**2 * (K0**2 + Gamma_negative**2))
Tk_total = Tk_up + Tk_down
S_z_stationary = (hbar / 2) * ((Tk_up - Tk_down) / (Tk_up + Tk_down))

t = np.linspace(0, 15000, 100000)
S_z_stationary_array = S_z_stationary + 0 * t

Psi_up = (
    (np.sqrt(2) / np.sqrt(Tk_total)) * (1 / np.sqrt(2)) *
    ((K0 - 1j * Gamma_positive) / (math.cosh(Alpha_positive) * (K0**2 + Gamma_positive**2)))
) * (
    K0 * moshinsky_function(x - a, K0, t, m, hbar) +
    1j * Gamma_positive * moshinsky_function(x - a, -1j * Gamma_positive, t, m, hbar)
)
Psi_up_density = np.real(Psi_up * np.conj(Psi_up))

Psi_down = (
    (np.sqrt(2) / np.sqrt(Tk_total)) * (1 / np.sqrt(2)) *
    ((K0 - 1j * Gamma_negative) / (math.cosh(Alpha_negative) * (K0**2 + Gamma_negative**2)))
) * (
    K0 * moshinsky_function(x - a, K0, t, m, hbar) +
    1j * Gamma_negative * moshinsky_function(x - a, -1j * Gamma_negative, t, m, hbar)
)
Psi_down_density = np.real(Psi_down * np.conj(Psi_down))
Psi_total = Psi_down_density + Psi_up_density

S_x = (hbar / 2) * (np.conj(Psi_down) * Psi_up + np.conj(Psi_up) * Psi_down)
S_y = 1j * (hbar / 2) * (np.conj(Psi_down) * Psi_up - np.conj(Psi_up) * Psi_down)
S_z = (hbar / 2) * (Psi_up_density - Psi_down_density)

MV_S_x = np.real(S_x)
MV_S_y = np.real(S_y)
MV_S_z = np.real(S_z)
MODS = (MV_S_x**2 + MV_S_y**2 + MV_S_z**2) * 4 / (hbar**2)

print(MODS)
print(Alpha)
v = (hbar / m) * Kv0
print(a / v)
print((hbar / 2) * omega * (a / v))
print(1 / Kv0, a)
print((x * m) / (hbar * Kv0))

fig = plt.figure(1)
ax = fig.add_axes([0, 0, 1, 1])
plt.grid()
line1, = plt.plot(t, MV_S_x, label='Expectation value of Sx')
plt.legend(handles=[line1], loc='upper right')
ax.set_title('Spin Expectation Values')
ax.set_xlabel('t (fs)')
ax.set_ylabel('Expectation Value')
plt.savefig('figure.png')

fig = plt.figure(2)
ax = fig.add_axes([0, 0, 1, 1])
plt.grid()
line2, = plt.plot(t, MV_S_y, label='Expectation value of Sy')
plt.legend(handles=[line2], loc='upper right')
ax.set_title('Spin Expectation Values')
ax.set_xlabel('t (fs)')
ax.set_ylabel('Expectation Value')
plt.savefig('figure.png')

fig = plt.figure(3)
ax = fig.add_axes([0, 0, 1, 1])
plt.grid()
line3, = plt.plot(t, MV_S_z, label='Expectation value of Sz')
line2, = plt.plot(t, S_z_stationary_array, label='Stationary Sz')
plt.legend(handles=[line3], loc='upper right')
ax.set_title('Spin Expectation Values')
ax.set_xlabel('t (fs)')
ax.set_ylabel('Expectation Value')
plt.savefig('figure.png')

fig = plt.figure(4)
ax = fig.add_axes([0, 0, 1, 1])
plt.grid()
line1, = plt.plot(t, MODS, label='Sum of squared expectation values')
plt.legend(handles=[line1], loc='upper right')
ax.set_title('Verification that the result equals one')
ax.set_xlabel('t (fs)')
ax.set_ylabel('$(\\langle S_x \\rangle^2 + \\langle S_y \\rangle^2 + \\langle S_z \\rangle^2)(4/\\hbar)$')
plt.savefig('figure.png')

fig = plt.figure(5)
ax = fig.add_axes([0, 0, 1, 1])
plt.grid()
line1, = plt.plot(t, 4.73e-8 * MODS, label='Sum of squared expectation values')
line2, = plt.plot(t, -1 * MV_S_y, label='Expectation value of Sz')
ax.set_title('Verification that the result equals one')
ax.set_xlabel('t (fs)')
ax.set_ylabel('$(\\langle S_x \\rangle^2 + \\langle S_y \\rangle^2 + \\langle S_z \\rangle^2)(4/\\hbar)$')
plt.savefig('figure.png')


/tmp/ipykernel_2878/1597770408.py:13: RuntimeWarning: invalid value encountered in divide
  (x - (hbar * k * t / m)) /


[       nan 2.9182505  2.83596295 ... 1.0060202  1.00602016 1.00602013]
20.952942404882144
68.95738015454022
5.219685344212354
23.862996916533184 500
68.95738015454023
